# MR Annotation Accuracy

Evaluate medication recommendation annotation accuracy from `MR_reviewed.xlsx`.

The reviewed workbook is expected to have one sheet per document, with medication annotations under `### 医薬品 ###` and correctness labels under `### 正誤 ###`. Rows marked `正` are treated as correct, rows marked `誤` are treated as incorrect, and blank labels are reported as pending/unreviewed.

In [1]:
from pathlib import Path
import ast

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = Path("MR_reviewed.xlsx")
BENCHMARK_PATH = Path("data/benchmarks/CLR/MR/data.csv")

WORKBOOK_PATH.exists(), BENCHMARK_PATH.exists()

(True, True)

In [2]:
def normalize_cell(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def find_header(ws):
    for row in ws.iter_rows():
        values = [cell.value for cell in row]
        if "### 医薬品 ###" in values and "### 正誤 ###" in values:
            header_row = row[0].row
            columns = {
                ws.cell(header_row, col).value: col
                for col in range(1, ws.max_column + 1)
                if ws.cell(header_row, col).value is not None
            }
            return header_row, columns
    raise ValueError(f"Header row not found in sheet: {ws.title}")


def load_reviewed_workbook(path):
    wb = load_workbook(path, data_only=True, read_only=False)
    records = []

    for ws in wb.worksheets:
        header_row, columns = find_header(ws)
        medra_col = columns.get("### MedDRA ###")
        medication_col = columns["### 医薬品 ###"]
        label_col = columns["### 正誤 ###"]
        comment_col = label_col + 1 if label_col + 1 <= ws.max_column else None

        for row_idx in range(header_row + 1, ws.max_row + 1):
            medication = normalize_cell(ws.cell(row_idx, medication_col).value)
            label = normalize_cell(ws.cell(row_idx, label_col).value)
            medra = normalize_cell(ws.cell(row_idx, medra_col).value) if medra_col else None
            comment = normalize_cell(ws.cell(row_idx, comment_col).value) if comment_col else None

            if medication is None and label is None and medra is None and comment is None:
                continue

            records.append({
                "doc_name": ws.title,
                "row": row_idx,
                "text": ws["A1"].value,
                "meddra": medra,
                "medication": medication,
                "label": label,
                "comment": comment,
            })

    return pd.DataFrame(records)


review_df = load_reviewed_workbook(WORKBOOK_PATH)
review_df.head()

,doc_name,row,text,meddra,medication,label,comment
0,12_118B35_緊張性気胸_p2,3,S やっぱり薬切れると痛いですね。ちょっといきむとぼこぼこ出てきます。\nO BP 125/...,肋骨骨折,アセトアミノフェン,正,None
1,12_118B35_緊張性気胸_p2,4,S やっぱり薬切れると痛いですね。ちょっといきむとぼこぼこ出てきます。\nO BP 125/...,気胸,オキシコドン,誤,非がんの場合，慢性疼痛にのみ適応あり
2,12_118B35_緊張性気胸_p2,5,S やっぱり薬切れると痛いですね。ちょっといきむとぼこぼこ出てきます。\nO BP 125/...,疼痛,ロキソプロフェンナトリウム水和物,正,None
3,12_118B35_緊張性気胸_p2,6,S やっぱり薬切れると痛いですね。ちょっといきむとぼこぼこ出てきます。\nO BP 125/...,肺のエアリーク,コデインリン酸塩水和物,正,None
4,12_118B35_緊張性気胸_p2,7,S やっぱり薬切れると痛いですね。ちょっといきむとぼこぼこ出てきます。\nO BP 125/...,None,モルヒネ塩酸塩水和物,正,None


In [3]:
label_counts = review_df["label"].fillna("<blank>").value_counts(dropna=False)

summary = pd.DataFrame({
    "metric": [
        "sheets/documents",
        "annotation rows",
        "medication rows",
        "evaluated medication rows",
        "correct (正)",
        "incorrect (誤)",
        "blank / pending",
    ],
    "value": [
        review_df["doc_name"].nunique(),
        len(review_df),
        review_df["medication"].notna().sum(),
        review_df["label"].isin(["正", "誤"]).sum(),
        (review_df["label"] == "正").sum(),
        (review_df["label"] == "誤").sum(),
        review_df["label"].isna().sum(),
    ],
})

display(summary)
display(label_counts.rename_axis("label").reset_index(name="count"))

,metric,value
0,sheets/documents,30
1,annotation rows,496
2,medication rows,437
3,evaluated medication rows,402
4,correct (正),370
5,incorrect (誤),32
6,blank / pending,94


,label,count
0,正,370
1,<blank>,94
2,誤,32


In [4]:
evaluated = review_df[review_df["medication"].notna() & review_df["label"].isin(["正", "誤"])].copy()
pending = review_df[review_df["medication"].notna() & review_df["label"].isna()].copy()

correct = (evaluated["label"] == "正").sum()
incorrect = (evaluated["label"] == "誤").sum()
evaluated_total = len(evaluated)
all_medication_total = review_df["medication"].notna().sum()

overall_accuracy = correct / evaluated_total if evaluated_total else float("nan")
strict_accuracy = correct / all_medication_total if all_medication_total else float("nan")

accuracy_summary = pd.DataFrame([
    {
        "score": "accuracy_excluding_blank_labels",
        "correct": correct,
        "denominator": evaluated_total,
        "accuracy": overall_accuracy,
    },
    {
        "score": "strict_accuracy_blank_as_incorrect_or_pending",
        "correct": correct,
        "denominator": all_medication_total,
        "accuracy": strict_accuracy,
    },
])

accuracy_summary

,score,correct,denominator,accuracy
0,accuracy_excluding_blank_labels,370,402,0.920398
1,strict_accuracy_blank_as_incorrect_or_pending,370,437,0.846682


In [5]:
per_doc = (
    review_df[review_df["medication"].notna()]
    .assign(
        correct=lambda df: df["label"].eq("正"),
        incorrect=lambda df: df["label"].eq("誤"),
        pending=lambda df: df["label"].isna(),
        evaluated=lambda df: df["label"].isin(["正", "誤"]),
    )
    .groupby("doc_name", as_index=False)
    .agg(
        medication_rows=("medication", "size"),
        evaluated=("evaluated", "sum"),
        correct=("correct", "sum"),
        incorrect=("incorrect", "sum"),
        pending=("pending", "sum"),
    )
)

per_doc["accuracy_excluding_blank"] = per_doc["correct"] / per_doc["evaluated"].replace(0, pd.NA)
per_doc["strict_accuracy"] = per_doc["correct"] / per_doc["medication_rows"].replace(0, pd.NA)

per_doc.sort_values(["accuracy_excluding_blank", "incorrect", "pending"], ascending=[True, False, False]).head(20)

,doc_name,medication_rows,evaluated,correct,incorrect,pending,accuracy_excluding_blank,strict_accuracy
28,19_118D57_慢性硬膜下血腫_p2,3,3,0,3,0,0.000000,0.000000
10,10_118A69_胆管結石胆管炎_p1,19,19,11,8,0,0.578947,0.578947
2,06_118F64-66_胸髄損傷_p2,28,28,19,9,0,0.678571,0.678571
19,15_118C50_腎損傷_p3,10,10,9,1,0,0.900000,0.900000
12,118F73-75_AMI,81,81,75,6,0,0.925926,0.925926
15,12_118B35_緊張性気胸_p2,16,15,14,1,1,0.933333,0.875000
13,118F73-75_AMI-J,81,63,60,3,18,0.952381,0.740741
20,15_118C50_腎損傷_p4,41,41,40,1,0,0.975610,0.975610
29,20_118F56_高エネルギー外傷_p2,26,20,20,0,6,1.000000,0.769231
7,09_118A19_肺血栓塞栓症_p1,5,2,2,0,3,1.000000,0.400000


In [6]:
incorrect_rows = evaluated[evaluated["label"].eq("誤")].copy()
incorrect_rows[["doc_name", "row", "meddra", "medication", "comment"]].sort_values(["doc_name", "row"])

,doc_name,row,meddra,medication,comment
152,06_118F64-66_胸髄損傷_p2,3,感覚障害,イミプラミン塩酸塩,None
153,06_118F64-66_胸髄損傷_p2,4,脊髄損傷,バクロフェン,None
156,06_118F64-66_胸髄損傷_p2,7,単麻痺,エペリゾン塩酸塩,None
158,06_118F64-66_胸髄損傷_p2,9,None,プレガバリン,None
160,06_118F64-66_胸髄損傷_p2,11,None,チザニジン塩酸塩,None
163,06_118F64-66_胸髄損傷_p2,14,None,ガバペンチン,None
169,06_118F64-66_胸髄損傷_p2,20,None,アミトリプチリン塩酸塩,None
172,06_118F64-66_胸髄損傷_p2,23,None,ダントロレンナトリウム水和物,None
175,06_118F64-66_胸髄損傷_p2,26,None,ノルトリプチリン塩酸塩,None
126,10_118A69_胆管結石胆管炎_p1,3,上腹部痛,酢酸リンゲル液,この症例が近い未来必要になる，という意味では正


In [7]:
pending[["doc_name", "row", "meddra", "medication", "comment"]].sort_values(["doc_name", "row"])

,doc_name,row,meddra,medication,comment
301,09_118A19_肺血栓塞栓症,7,None,ヘパリンナトリウム,追加）サマリ内に記載があるが提案されていない？
320,09_118A19_肺血栓塞栓症_p1,5,肺塞栓症,アピキサバン,追加
321,09_118A19_肺血栓塞栓症_p1,6,深部静脈血栓症,エドキサバントシル酸塩水和物,追加
322,09_118A19_肺血栓塞栓症_p1,7,None,リバーロキサバン,追加
326,09_118A19_肺血栓塞栓症_p4,6,None,ワルファリンカリウム,追加
330,117C66-68_IE-F,6,いびき,ダプトマイシン,MRSAかどうか定かではない
332,117C66-68_IE-F,8,浮動性めまい,バンコマイシン塩酸塩,同上
333,117C66-68_IE-F,9,悪寒,ニカルジピン,追加
344,118F73-75_AMI-J,10,None,ベタキソロール塩酸塩,β遮断薬が第一選択ではない
348,118F73-75_AMI-J,14,None,アミオダロン塩酸塩,AMI後に不整脈があれば正
